In [1]:
import os
from osgeo import gdal

def export_sar_intensity(pol, input_dir, output_tif):
    """
    Extracts SAR magnitude (Band 1) from topophase.cor.geo,
    computes SAR intensity (magnitude squared), and exports it as a GeoTIFF.
    """
    cor_geo_path = os.path.join(input_dir, 'topophase.cor.geo')
    
    if not os.path.exists(cor_geo_path):
        print(f"Error: {cor_geo_path} does not exist. Ensure the original script completed successfully for {pol}.")
        return

    # Open the geocoded coherence/magnitude file
    ds = gdal.Open(cor_geo_path, gdal.GA_ReadOnly)
    if ds is None:
        print(f"Failed to open {cor_geo_path}")
        return
    
    # Band 1 contains the SAR magnitude/amplitude
    band1 = ds.GetRasterBand(1)
    magnitude = band1.ReadAsArray()
    
    # Calculate Intensity (Intensity = Magnitude^2)
    intensity = magnitude ** 2
    
    # Retrieve spatial information to preserve georeferencing
    geotransform = ds.GetGeoTransform()
    projection = ds.GetProjection()
    
    # Create the output GeoTIFF file
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(output_tif, ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Float32)
    
    # Apply georeferencing metadata to the new file
    out_ds.SetGeoTransform(geotransform)
    out_ds.SetProjection(projection)
    
    # Write the intensity data array
    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(intensity)
    
    # Set NoData value to 0.0 (standard for background pixels in ISCE)
    out_band.SetNoDataValue(0.0)
    
    # Save and close datasets
    out_band.FlushCache()
    out_ds = None
    ds = None
    print(f"Successfully exported {pol} intensity to: {output_tif}")

# Target both polarization folders generated by your original script
polarizations = ['vv', 'vh']

for pol in polarizations:
    input_directory = f'merged_{pol}'
    output_filename = f'sar_intensity_{pol}.tif'
    export_sar_intensity(pol, input_directory, output_filename)

ERROR! Session/line number was not unique in database. History logging moved to new session 217


ERROR 1: PROJ: proj_create_from_database: Open of /home/st-juho/code_testing/miniconda3/envs/env39/share/proj failed


Successfully exported vv intensity to: sar_intensity_vv.tif
Successfully exported vh intensity to: sar_intensity_vh.tif


ERROR 1: PROJ: proj_create_from_database: Open of /home/st-juho/code_testing/miniconda3/envs/env39/share/proj failed
